In [ ]:
# Install required packages
!pip install torch>=2.0.0 transformers>=4.36.0 datasets>=2.14.0 accelerate>=0.24.0
!pip install bitsandbytes>=0.41.0 flash-attn>=2.3.0 huggingface-hub>=0.19.0
!pip install peft>=0.7.0 trl>=0.7.0 numpy>=1.24.0 scipy>=1.10.0
!pip install scikit-learn>=1.3.0 tqdm>=4.65.0 sentencepiece>=0.1.99 protobuf>=3.20.0

In [ ]:
# Import required libraries
import os
import json
import torch
import logging
from typing import Dict, List, Any
from dataclasses import dataclass
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import numpy as np
from huggingface_hub import login

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
@dataclass
class TrainingConfig:
    """Configuration for model training"""
    # Model settings
    base_model: str = "Qwen/Qwen2.5-7B-Instruct"
    model_name: str = "bi-intent-discovery-qwen"
    
    # Training settings
    num_epochs: int = 3
    batch_size: int = 2  # Reduced for Colab
    gradient_accumulation_steps: int = 8  # Increased for Colab
    learning_rate: float = 2e-5
    warmup_steps: int = 100
    max_seq_length: int = 2048
    
    # Data settings
    train_data_path: str = "training_data_500_examples.json"
    prompt_template_path: str = "resources/prompts/prompt_new.txt"
    
    # Output settings
    output_dir: str = "./trained_model"
    save_to_hf: bool = True
    hf_username: str = None  # Set your HF username here
    
    # Hardware settings (optimized for Colab)
    use_4bit: bool = True
    use_8bit: bool = False
    use_flash_attention: bool = False  # Disabled for Colab compatibility

# Create config instance
config = TrainingConfig()

# Set your Hugging Face username here
config.hf_username = "ssuki"  # Replace with your actual username

print("Training Configuration:")
print(f"Base Model: {config.base_model}")
print(f"Training Epochs: {config.num_epochs}")
print(f"Batch Size: {config.batch_size}")
print(f"Learning Rate: {config.learning_rate}")
print(f"Output Directory: {config.output_dir}")
print(f"Save to HF: {config.save_to_hf}")
if config.hf_username:
    print(f"HF Username: {config.hf_username}")

In [ ]:
from google.colab import files

print("Upload your training data file (training_data_500_examples.json):")
uploaded = files.upload()

print("\nUpload your prompt template file (prompt_new.txt):")
uploaded_prompt = files.upload()

# Create directories if needed
os.makedirs("resources/prompts", exist_ok=True)

# Move files to correct locations
if 'training_data_500_examples.json' in uploaded:
    with open('training_data_500_examples.json', 'wb') as f:
        f.write(uploaded['training_data_500_examples.json'])
    print("Training data uploaded successfully")

if 'prompt_new.txt' in uploaded_prompt:
    with open('resources/prompts/prompt_new.txt', 'wb') as f:
        f.write(uploaded_prompt['prompt_new.txt'])
    print("Prompt template uploaded successfully")

In [ ]:
class BIIntentTrainer:
    """Trainer for BI Intent Discovery model"""
    
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.tokenizer = None
        self.model = None
        self.trainer = None
        
    def load_prompt_template(self) -> str:
        """Load the prompt template"""
        try:
            with open(self.config.prompt_template_path, 'r', encoding='utf-8') as f:
                return f.read().strip()
        except FileNotFoundError:
            logger.warning(f"Prompt template not found at {self.config.prompt_template_path}")
            return self._get_default_prompt()
    
    def _get_default_prompt(self) -> str:
        """Default prompt template if file not found"""
        return """# BI Planning & Discovery Agent
You are an AI assistant specialized in analyzing natural language BI questions and breaking them into structured steps for query building.

## Phases
### Phase 1: Planning
- Detect if question is **complex** (multi-step, dependencies, ranking, comparison, or time-based logic).  
- Complexity indicators: "for the X", "top/best/highest/lowest X", "X that are Y", "based on X", "compare X with Y", "X for those Y".  
- If complex:  
  1. Extract BI elements (measures, dimensions, time, filters).  
  2. Break into ordered steps (like CTEs).  
  3. Add post-processing (ranking, sorting, formatting).  
- If simple: skip planning.  

### Phase 2: Discovery
For each question or planning step:  
1. Extract BI concepts (measures, dimensions, timeframes, timegrain, patterns, filters, segments, breakdowns).  
2. Map exact phrases to BI fields (store in `original_phrase`).  
3. Capture **all unmatched terms** in `unmatched_intents` with `phrase`, `type`, and `reason`.  
4. Handle **ambiguity**: If a phrase can mean multiple things, request clarification.  

## Output Format
Respond with a JSON object containing your intent and discovery results.

## Question: {question}

## Response:"""

In [ ]:
def load_training_data(self) -> List[Dict[str, Any]]:
        """Load and preprocess training data"""
        logger.info(f"Loading training data from {self.config.train_data_path}")
        
        try:
            with open(self.config.train_data_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except FileNotFoundError:
            raise FileNotFoundError(f"Training data not found at {self.config.train_data_path}")
        
        logger.info(f"Loaded {len(data)} training examples")
        return data
    
def format_training_example(self, example: Dict[str, Any], prompt_template: str) -> str:
        """Format a training example into the model's expected format"""
        question = example["input"]
        expected_output = json.dumps(example["output"], ensure_ascii=False, indent=2)
        
        # Format the prompt
        formatted_prompt = prompt_template.format(question=question)
        
        # Create the full training text
        training_text = f"{formatted_prompt}\n{expected_output}"
        
        return training_text
    
def prepare_dataset(self, data: List[Dict[str, Any]], prompt_template: str) -> Dataset:
        """Prepare the dataset for training"""
        logger.info("Preparing dataset...")
        
        formatted_examples = []
        for example in data:
            try:
                formatted_text = self.format_training_example(example, prompt_template)
                formatted_examples.append({"text": formatted_text})
            except Exception as e:
                logger.warning(f"Error formatting example: {e}")
                continue
        
        logger.info(f"Successfully formatted {len(formatted_examples)} examples")
        
        # Create dat

In [ ]:
def load_model_and_tokenizer(self):
        """Load the base model and tokenizer"""
        logger.info(f"Loading model: {self.config.base_model}")
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.base_model,
            trust_remote_code=True,
            padding_side="right"
        )
        
        # Add padding token if not present
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Load model with optimizations
        model_kwargs = {
            "trust_remote_code": True,
            "torch_dtype": torch.float16,
        }
        
        if self.config.use_4bit:
            model_kwargs.update({
                "load_in_4bit": True,
                "quantization_config": {
                    "load_in_4bit": True,
                    "bnb_4bit_compute_dtype": torch.float16,
                    "bnb_4bit_use_double_quant": True,
                    "bnb_4bit_quant_type": "nf4"
                }
            })
        elif self.config.use_8bit:
            model_kwargs["load_in_8bit"] = True
        
        if self.config.use_flash_attention:
            model_kwargs["attn_implementation"] = "flash_attention_2"
        
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model,
            **model_kwargs
        )
        
        # Enable gradient checkpointing for memory efficiency
        self.model.gradient_checkpointing_enable()
        
        logger.info("Model and tokenizer loaded successfully")
    
def tokenize_function(self, examples):
        """Tokenize the dataset"""
        return self.tokenizer(
            examples["text"],
            truncation=True,
            padding=True,
            max_length=self.config.max_seq_length,
            return_tensors="pt"
        )

In [ ]:
def setup_training(self, dataset: Dataset):
        """Setup the training configuration"""
        logger.info("Setting up training...")
        
        # Tokenize dataset
        tokenized_dataset = dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=dataset.column_names
        )
        
        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir=self.config.output_dir,
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            learning_rate=self.config.learning_rate,
            warmup_steps=self.config.warmup_steps,
            logging_steps=10,
            save_steps=500,
            eval_steps=500,
            evaluation_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            fp16=True,
            dataloader_pin_memory=False,
            remove_unused_columns=False,
            report_to=None,  # Disable wandb/tensorboard
        )
        
        # Initialize trainer
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_dataset,
            eval_dataset=tokenized_dataset.select(range(min(100, len(tokenized_dataset)))),
            data_collator=data_collator,
            tokenizer=self.tokenizer,
        )
        
        logger.info("Training setup completed")

In [ ]:
def train(self):
        """Execute the training process"""
        logger.info("Starting training...")
        
        # Load prompt template
        prompt_template = self.load_prompt_template()
        
        # Load and prepare data
        raw_data = self.load_training_data()
        dataset = self.prepare_dataset(raw_data, prompt_template)
        
        # Load model and tokenizer
        self.load_model_and_tokenizer()
        
        # Setup training
        self.setup_training(dataset)
        
        # Start training
        logger.info("Training started...")
        train_result = self.trainer.train()
        
        # Save the model
        logger.info("Saving model...")
        self.trainer.save_model()
        self.tokenizer.save_pretrained(self.config.output_dir)
        
        # Save training metrics
        metrics = train_result.metrics
        with open(os.path.join(self.config.output_dir, "training_metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)
        
        logger.info(f"Training completed. Metrics: {metrics}")
        
        return train_result

In [ ]:
def save_to_huggingface(self):
        """Save the trained model to Hugging Face Hub"""
        if not self.config.save_to_hf:
            logger.info("Skipping Hugging Face upload (save_to_hf=False)")
            return
        
        if not self.config.hf_username:
            logger.warning("HF username not provided, skipping upload")
            return
        
        try:
            # Login to Hugging Face
            login()
            
            # Model name for HF
            model_name = f"{self.config.hf_username}/{self.config.model_name}"
            
            logger.info(f"Uploading model to Hugging Face: {model_name}")
            
            # Push model and tokenizer
            self.model.push_to_hub(model_name)
            self.tokenizer.push_to_hub(model_name)
            
            # Create model card
            self._create_model_card(model_name)
            
            logger.info(f"Model successfully uploaded to: https://huggingface.co/{model_name}")
            
        except Exception as e:
            logger.error(f"Error uploading to Hugging Face: {e}")
    
def _create_model_card(self, model_name: str):
        """Create a model card for the uploaded model"""
        model_card = f"""---
language:
- en
tags:
- bi-intent-discovery
- business-intelligence
- question-analysis
- structured-output
license: mit
---

# BI Intent Discovery Model

This model is fine-tuned from Qwen2.5-7B-Instruct to perform Business Intelligence (BI) intent discovery tasks.

## Model Description

The model analyzes natural language questions about business intelligence data and breaks them down into structured steps for query building. It performs two main phases:

1. **Planning Phase**: Detects complex questions and breaks them into ordered steps
2. **Discovery Phase**: Extracts BI concepts (measures, dimensions, timeframes, etc.) from questions

## Training Data

- 500 examples of BI questions with structured outputs
- Covers various complexity levels from simple to multi-step queries
- Includes examples with ambiguity handling and unmatched intent capture

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

# Load model
model_name = "{model_name}"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

# Example usage
question = "Show me total sales by region for the last quarter"
# Format with your prompt template and generate response
```

## Output Format

The model outputs structured JSON containing:
- Intent classification
- Discovery results with measures, dimensions, timeframes
- Unmatched intents for ambiguous terms
- Step-by-step breakdown for complex queries

## Training Configuration

- Base Model: Qwen2.5-7B-Instruct
- Training Examples: 500
- Epochs: {self.config.num_epochs}
- Learning Rate: {self.config.learning_rate}
- Max Sequence Length: {self.config.max_seq_length}
"""
        
        # Save model card
        card_path = os.path.join(self.config.output_dir, "README.md")
        with open(card_path, "w", encoding="utf-8") as f:
            f.write(model_card)
        
        # Upload model card
        try:
            from huggingface_hub import upload_file
            upload_file(
                path_or_fileobj=card_path,
                path_in_repo="README.md",
                repo_id=model_name,
                repo_type="model"
            )
        except Exception as e:
            logger.warning(f"Could not upload model card: {e}")

In [ ]:
# Initialize trainer
trainer = BIIntentTrainer(config)

# Start training
print("Starting training process...")
train_result = trainer.train()

print("Training completed!")
print(f"Final training loss: {train_result.training_loss:.4f}")

In [ ]:
# Save to Hugging Face
if config.save_to_hf and config.hf_username:
    print("Uploading model to Hugging Face...")
    trainer.save_to_huggingface()
    print("Upload completed!")
else:
    print("Skipping Hugging Face upload")

In [ ]:
# Create a zip file of the trained model
import zipfile

def zip_model_files():
    """Create a zip file of the trained model"""
    zip_path = "trained_model.zip"
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(config.output_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, config.output_dir)
                zipf.write(file_path, arcname)
    
    return zip_path

# Create zip file
zip_path = zip_model_files()
print(f"Model files zipped to: {zip_path}")

# Download the zip file
from google.colab import files
files.download(zip_path)

In [ ]:
def test_model(question: str):
    """Test the trained model with a sample question"""
    # Load the trained model
    model_path = config.output_dir
    
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    # Load prompt template
    prompt_template = trainer.load_prompt_template()
    formatted_prompt = prompt_template.format(question=question)
    
    # Generate response
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the response part (after the prompt)
    response_text = response[len(formatted_prompt):].strip()
    
    return response_text

# Test with a sample question
test_question = "Show me total sales by region for the last quarter"
print(f"Testing with question: {test_question}")
print("=" * 50)

try:
    response = test_model(test_question)
    print(response)
except Exception as e:
    print(f"Error testing model: {e}")